In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tqdm import tqdm
from tensorflow.keras.models import load_model


In [2]:
DATA_DIR = r"D:\SER_Cross\data\processed"
FEATURE_DIR = r"D:\SER_Cross\features"


In [3]:
df_train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
print(df_train["dataset"].value_counts())


dataset
crema_d    4408
ravdess     660
Name: count, dtype: int64


In [4]:
df_crema = df_train[df_train["dataset"] == "crema_d"]
print("CREMA-D samples:", len(df_crema))


CREMA-D samples: 4408


In [5]:
SAMPLE_RATE = 16000
N_MELS = 128
N_FFT = 400
HOP_LENGTH = 160
MAX_LEN = 300


In [6]:
emotion_to_label = {
    "neutral": 0,
    "happy": 1,
    "sad": 2,
    "angry": 3,
    "fear": 4
}


In [7]:
def extract_log_mel(wav_path):
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    log_mel = librosa.power_to_db(mel, ref=np.max)

    if log_mel.shape[1] < MAX_LEN:
        log_mel = np.pad(
            log_mel,
            ((0, 0), (0, MAX_LEN - log_mel.shape[1])),
            mode="constant"
        )
    else:
        log_mel = log_mel[:, :MAX_LEN]

    return log_mel


In [8]:
X_crema, y_crema = [], []

for _, row in tqdm(df_crema.iterrows(), total=len(df_crema)):
    X_crema.append(extract_log_mel(row["path"]))
    y_crema.append(emotion_to_label[row["emotion"]])

X_crema = np.array(X_crema)[..., np.newaxis]
y_crema = np.array(y_crema)

print(X_crema.shape, y_crema.shape)


  0%|          | 0/4408 [00:00<?, ?it/s]c:\Users\Asus\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\Asus\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Users\Asus\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
100%|██████████| 4408/4408 [00:33<00:00, 132.36it/s]


(4408, 128, 300, 1) (4408,)


In [10]:
from tensorflow.keras.layers import Layer

class Attention(Layer):
    def __init__(self):
        super().__init__()

    def call(self, inputs):
        score = tf.nn.softmax(tf.reduce_mean(inputs, axis=-1), axis=1)
        score = tf.expand_dims(score, axis=-1)
        context = tf.reduce_sum(inputs * score, axis=1)
        return context


In [12]:
from tensorflow.keras.layers import Layer
import tensorflow as tf

class Attention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        # inputs: (batch, time, features)
        score = tf.nn.softmax(tf.reduce_mean(inputs, axis=-1), axis=1)
        score = tf.expand_dims(score, axis=-1)
        context = tf.reduce_sum(inputs * score, axis=1)
        return context

    def get_config(self):
        config = super().get_config()
        return config


In [13]:
model = load_model(
    "best_ser_model.h5",
    custom_objects={"Attention": Attention},
    compile=False
)


In [14]:
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 300, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 300, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 150, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 150, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 150, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 75, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 32, 4800)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 32, 256)        │     5,047,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_1 (Attention)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,100,037 (19.46 MB)

 Trainable params: 5,099,845 (19.45 MB)

 Non-trainable params: 192 (768.00 B)

In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [17]:
model.fit(
    X_crema, y_crema,
    epochs=15,
    batch_size=32,
    validation_split=0.1
)


Epoch 1/15
124/124 ━━━━━━━━━━━━━━━━━━━━ 66s 481ms/step - accuracy: 0.9120 - loss: 0.3096 - val_accuracy: 0.9773 - val_loss: 0.1778
Epoch 2/15
124/124 ━━━━━━━━━━━━━━━━━━━━ 58s 467ms/step - accuracy: 0.9458 - loss: 0.2047 - val_accuracy: 0.9433 - val_loss: 0.2117
Epoch 3/15
124/124 ━━━━━━━━━━━━━━━━━━━━ 58s 467ms/step - accuracy: 0.9703 - loss: 0.1354 - val_accuracy: 0.9592 - val_loss: 0.1692
Epoch 4/15
124/124 ━━━━━━━━━━━━━━━━━━━━ 58s 468ms/step - accuracy: 0.9816 - loss: 0.0950 - val_accuracy: 0.9252 - val_loss: 0.2254
Epoch 5/15
124/124 ━━━━━━━━━━━━━━━━━━━━ 59s 478ms/step - accuracy: 0.9902 - loss: 0.0675 - val_accuracy: 0.9410 - val_loss: 0.1992
Epoch 6/15
124/124 ━━━━━━━━━━━━━━━━━━━━ 60s 481ms/step - accuracy: 0.9990 - loss: 0.0390 - val_accuracy: 0.9388 - val_loss: 0.1811
Epoch 7/15
124/124 ━━━━━━━━━━━━━━━━━━━━ 60s 482ms/step - accuracy: 0.9980 - loss: 0.0255 - val_accuracy: 0.9410 - val_loss: 0.1879
Epoch 8/15
124/124 ━━━━━━━━━━━━━━━━━━━━ 60s 480ms/step - accuracy: 0.9950 - loss: 0

In [18]:
model.optimizer


In [19]:
df_rav = df_train[df_train["dataset"] == "ravdess"]
print("RAVDESS samples:", len(df_rav))


RAVDESS samples: 660


In [20]:
X_rav, y_rav = [], []

for _, row in tqdm(df_rav.iterrows(), total=len(df_rav)):
    X_rav.append(extract_log_mel(row["path"]))
    y_rav.append(emotion_to_label[row["emotion"]])

X_rav = np.array(X_rav)[..., np.newaxis]
y_rav = np.array(y_rav)

print(X_rav.shape, y_rav.shape)


100%|██████████| 660/660 [00:04<00:00, 139.05it/s]


(660, 128, 300, 1) (660,)


In [21]:
for layer in model.layers:
    if "conv" in layer.name.lower():
        layer.trainable = False


In [22]:
[(layer.name, layer.trainable) for layer in model.layers]


[('input_layer', True),
 ('conv2d', False),
 ('max_pooling2d', True),
 ('batch_normalization', True),
 ('conv2d_1', False),
 ('max_pooling2d_1', True),
 ('batch_normalization_1', True),
 ('reshape', True),
 ('bidirectional', True),
 ('attention_1', True),
 ('dense', True),
 ('dropout', True),
 ('dense_1', True)]

In [23]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [24]:
history_ft = model.fit(
    X_rav, y_rav,
    epochs=20,
    batch_size=16,
    validation_split=0.2
)


Epoch 1/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 16s 297ms/step - accuracy: 0.7500 - loss: 0.7497 - val_accuracy: 0.8864 - val_loss: 0.3153
Epoch 2/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 265ms/step - accuracy: 0.8182 - loss: 0.4888 - val_accuracy: 0.8788 - val_loss: 0.2957
Epoch 3/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 259ms/step - accuracy: 0.8788 - loss: 0.3631 - val_accuracy: 0.8939 - val_loss: 0.2990
Epoch 4/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 256ms/step - accuracy: 0.8939 - loss: 0.2848 - val_accuracy: 0.8864 - val_loss: 0.2927
Epoch 5/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 262ms/step - accuracy: 0.9280 - loss: 0.2093 - val_accuracy: 0.8788 - val_loss: 0.3132
Epoch 6/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 259ms/step - accuracy: 0.9394 - loss: 0.1944 - val_accuracy: 0.8788 - val_loss: 0.3166
Epoch 7/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 253ms/step - accuracy: 0.9299 - loss: 0.1595 - val_accuracy: 0.8864 - val_loss: 0.2952
Epoch 8/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 245ms/step - accuracy: 0.9508 - loss: 0.1530 - val_accuracy: 0

In [25]:
model.save("final_finetuned_ser_model.h5")


In [26]:
X_test = np.load(r"D:\SER_Cross\features\test\X.npy")[..., np.newaxis]
y_test = np.load(r"D:\SER_Cross\features\test\y.npy")

test_loss, test_acc = model.evaluate(X_test, y_test)
print("Final Test Accuracy:", test_acc)


36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 120ms/step - accuracy: 0.4566 - loss: 2.6939
Final Test Accuracy: 0.45656028389930725
